In [ ]:
## Авторство
## Данный проект разработан и принадлежит ООО «Карпов Курсы».  
## Официальный сайт: https://karpov.courses

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
### Подключение к базе данных
from sqlalchemy import create_engine

# Вставьте строку подключения вместо <ВСТАВИТЬ_DATABASE_URL_СЮДА>
engine = create_engine("<ВСТАВИТЬ_DATABASE_URL_СЮДА>")

# Раскомментировать для реального подключения:
# connection = engine.connect().execution_options(stream_results=True)

connection = None  # заглушка 

In [ ]:
user_data = pd.read_sql('SELECT * FROM public.user_data;', con=connection) 

In [ ]:
print(f'DataFrame shape: {user_data.shape}')
print(f'user_id nunique: {user_data.user_id.nunique()}')

In [ ]:
post_text_df = pd.read_sql('SELECT * FROM public.post_text_df;', con=connection) 

In [ ]:
print(f'DataFrame shape: {post_text_df.shape}')
print(f'post_id nunique: {post_text_df.post_id.nunique()}')

In [ ]:
source_feed_limit = 1000000
feed_data = pd.read_sql(f'SELECT * FROM public.feed_data LIMIT {source_feed_limit};', con=connection) 

In [8]:
post_feed_df = post_text_df.merge(feed_data, on='post_id')
result_df = user_data.merge(post_feed_df, on='user_id')

In [ ]:
result_df.isna().sum()

In [ ]:
print(f'DataFrame shape: {result_df.shape}')
print(f'post_id nunique: {result_df.post_id.nunique()}')
print(f'user_id nunique: {result_df.user_id.nunique()}')

In [12]:
df = result_df.copy()

In [ ]:
df.user_id = df.user_id.astype(str)
df.post_id = df.post_id.astype(str)
df = df.sort_values('timestamp')

In [ ]:
df['text_len'] = df['text'].apply(len) # новый столбец с длиной строк из 'text'

In [ ]:
### Разделение на трейн-тест

train = df.iloc[:-200000].copy()
test = df.iloc[200000:].copy()

In [ ]:
### сколько постов просмотрел юзер за тренировочный период
user_count_posts = train.groupby('user_id').size()
train['user_view_posts'] = train['user_id'].map(user_count_posts)

In [ ]:
### Среднее количество просмотров всех юзеров

overall_views_mean = user_count_posts.mean()

test['user_view_posts'] = (
    test['user_id']
    .map(user_count_posts)
    .fillna(overall_views_mean)
)

train = train.drop(['user_id', 'post_id', 'action','timestamp', 'text'], axis=1)
test = test.drop(['user_id', 'post_id', 'action', 'timestamp', 'text'], axis=1)

In [ ]:
X_train = train.drop('target', axis=1)
X_test = test.drop('target', axis=1)
y_train = train['target']
y_test = test['target']

In [ ]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier()

catboost.fit(X_train,
             y_train,
             cat_features=['gender', 'country', 'city', 'os', 'source', 'topic'],
)

In [28]:
y_pred = catboost.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
print(f1_score(y_test, y_pred, average='weighted'))

In [ ]:
catboost.save_model('control',
                    format="cbm")